In [1]:
from google.colab import auth
auth.authenticate_user()

In [50]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [51]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
FOLDER_PATH= "data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/"
DATASET_ID = "produccion"
TABLE_ID= "ERRORES_desgravamen_prestamos"
FECHA_PERIODO= "2025-06-16"


# SCRIPT COMPLETO

In [52]:
### CLIENTES DE STORAGE Y BIGQUERY
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

bucket = storage_client.bucket(BUCKET_NAME)
blob_errores = bucket.blob('data_entries/REPORTES DE ERRORES/tabla_errores2.xlsx')
df_errores= pd.read_excel(BytesIO(blob_errores.download_as_string()), sheet_name='Sheet1', dtype={'CODIGO_ERROR': str, 'IDEERROR': str})
df_errores.loc[df_errores['CODIGO_ERROR'].isna(), 'CODIGO_ERROR'] = ""
df_errores.loc[df_errores['IDEERROR'].isna(), 'IDEERROR'] = ""
#df_errores.drop(['IDEERROR'], axis=1, inplace=True)

schema_desgramen = [
        bigquery.SchemaField("NRO_LOTE", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("LOTES_ANTERIORES", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_PLAN", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("NOMBRE_DE_PLAN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("COD_DE_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECINI_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECFIN_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("IDEDET", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("TIPO_MOVIMIENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_FIN ", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SUMA_ASEGURADA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA_RECARGO", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMABRUTACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMANETACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("NOMCOMPLETO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEPATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEMATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECNACIMIENTO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_DE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("LINEA_TRAMA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("ORIGEN_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR_SAS", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_TRAMA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_PERIODO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_SEGURO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_MOVIMIENTO_NUM", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_REGISTRO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("IDEERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DETALLE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION_2", bigquery.enums.SqlTypeNames.STRING)
        ]
#### FUNCION PARA LIMPIAR Y TRANSFORMAR LA COLUMNA A UN FORMATO DE FECHA
def limpiar_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

#### FUNCION PARA EXTRAER LA FECHA DEL NOMBRE DE ARCHIVO
def extraer_fecha(fecha):
    try:
        fecha_str = fecha.split("_")[2]
        return datetime.strptime(fecha_str, "%Y%m%d").date()
    except Exception:
        return None  # En caso de error

#### FUNCION PARA CLASIFICAR EL REGISTRO
def clasificar(row):
    # Si lote no es nulo
    if str(row['LOTES_ANTERIORES']).strip() != "":
        return 'Regularizacion no Exitosa'

    # Si último número del nombre de archivo > 100
    try:
        ultimo_numero = int(row['NOMBRE_DE_ARCHIVO'].split("_")[-1][:-4])
        if ultimo_numero > 100:
            return 'Regularizacion no Exitosa'
    except:
        pass

    # Si ninguna condición se cumple
    return 'Mes corriente'

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f' ----- Se ha creado la tabla "{table_id}" en el dataset "{dataset_id}" -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    #print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return


### LISTAR LOS ARCHIVOS QUE ESTAN DENTRO DEL BUCKET
bucket = storage_client.bucket(BUCKET_NAME)
blobs_excels = list(bucket.list_blobs(prefix=FOLDER_PATH))

### CICLO POR LA LISTA DE ARCHIVOS EXCEL
for blob in blobs_excels:
  if blob.name.endswith(".xlsx"):
    print(f"..... CARGANDO ARCHIVO: {blob.name} ....")
    file = blob.download_as_string()
    df_desgravamen= pd.read_excel(BytesIO(file), sheet_name='Hoja1', dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str,
                                                                             'IDELOTE': str, 'CODIGO ERROR': str,
                                                                             'SUMA ASEGURADA':str, 'TASA': str,
                                                                             'TASA RECARGO': str, 'PRIMABRUTACAN':str, 'PRIMANETACAN': str})
    ###### LIMPIEZA Y TRANSFORMACIONES

    # Colocar _ en los espacio de los nombres de las columnas
    df_desgravamen.columns = (df_desgravamen.columns.str.strip()
                                                    .str.upper()  # opcional: todo en mayúsculas
                                                    .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar estapacios por _
                              )
    # Borrar Columnas repetida
    df_desgravamen.drop(['IDEDET_1','ORIGEN_ERROR','UNNAMED__32'], axis=1, inplace=True)
    df_desgravamen= df_desgravamen.rename(columns={'IDELOTE': 'LINEA_TRAMA','CODIGO_ERROR':'ORIGEN_ERROR', 'DESCRIPCION_ERROR': 'CODIGO_ERROR'})

    # Imputar valores nulos
    df_desgravamen["MONEDA"] = df_desgravamen["MONEDA"].replace(r"^\s*$", None, regex=True)
    df_desgravamen["MONEDA"] = df_desgravamen["MONEDA"].where(df_desgravamen["MONEDA"].isin(["USD", "SOL"]), "SIN DATO")
    df_desgravamen= df_desgravamen.fillna({'LOTES_ANTERIORES':'', 'ORIGEN_ERROR': 'SIN DATO', 'TIPDOCUMENTO':'SIN DATO'})
    df_desgravamen.loc[df_desgravamen['ORIGEN_ERROR'] == 'nan', 'ORIGEN_ERROR'] = 'SIN DATO'
    # Cambiar el sepador de decimales
    df_desgravamen['SUMA_ASEGURADA'] = df_desgravamen['SUMA_ASEGURADA'].str.replace(',', '.', regex=False)
    df_desgravamen['TASA'] = df_desgravamen['TASA'].str.replace(',', '.', regex=False)
    df_desgravamen['TASA_RECARGO'] = df_desgravamen['TASA_RECARGO'].str.replace(',', '.', regex=False)
    df_desgravamen['PRIMABRUTACAN'] = df_desgravamen['PRIMABRUTACAN'].str.replace(',', '.', regex=False)
    df_desgravamen['PRIMANETACAN'] = df_desgravamen['PRIMANETACAN'].str.replace(',', '.', regex=False)

    #Cambiar el tipo de datos de las columnas
    df_desgravamen["LOTES_ANTERIORES"] = df_desgravamen["LOTES_ANTERIORES"].astype(str)
    df_desgravamen['FECHA_CARGA'] = pd.to_datetime(df_desgravamen['FECHA_CARGA'], format="%d/%m/%Y %I:%M:%S %p", errors='coerce')
    df_desgravamen['CODIGO_PRODUCTO'] = df_desgravamen['CODIGO_PRODUCTO'].fillna(0).astype('int')
    df_desgravamen["PRODUCTO"] = df_desgravamen["PRODUCTO"].astype(str)
    df_desgravamen['CODIGO_PLAN'] = df_desgravamen['CODIGO_PLAN'].fillna(0).astype('int')
    df_desgravamen["NOMBRE_DE_PLAN"] = df_desgravamen["NOMBRE_DE_PLAN"].astype(str)
    df_desgravamen["COD_DE_CERTIFICADO"] = df_desgravamen["COD_DE_CERTIFICADO"].astype(str)
    df_desgravamen['FECINI_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECINI_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen['FECFIN_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECFIN_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen["TIPO_MOVIMIENTO"] = df_desgravamen["TIPO_MOVIMIENTO"].astype(str)
    df_desgravamen['FEC__INICIO'] = pd.to_datetime(df_desgravamen['FEC__INICIO'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen['FEC__FIN'] = pd.to_datetime(df_desgravamen['FEC__FIN'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen['MONEDA'] = df_desgravamen['MONEDA'].astype(str)
    df_desgravamen['SUMA_ASEGURADA'] = pd.to_numeric(df_desgravamen['SUMA_ASEGURADA'], errors="coerce").astype('float64')
    df_desgravamen['TASA'] = pd.to_numeric(df_desgravamen['TASA'], errors="coerce").astype('float64')
    df_desgravamen['TASA_RECARGO'] = pd.to_numeric(df_desgravamen['TASA_RECARGO'], errors="coerce").astype('float64')
    df_desgravamen['PRIMABRUTACAN'] = pd.to_numeric(df_desgravamen['PRIMABRUTACAN'], errors="coerce").astype('float64')
    df_desgravamen['PRIMANETACAN'] = pd.to_numeric(df_desgravamen['PRIMANETACAN'], errors="coerce").astype('float64')
    df_desgravamen['NOMCOMPLETO'] = df_desgravamen['NOMCOMPLETO'].astype(str)
    df_desgravamen['APEPATERNO'] = df_desgravamen['APEPATERNO'].astype(str)
    df_desgravamen['APEMATERNO'] = df_desgravamen['APEMATERNO'].astype(str)
    df_desgravamen["FECNACIMIENTO"] = limpiar_fecha(df_desgravamen["FECNACIMIENTO"])
    df_desgravamen["TIPDOCUMENTO"] = df_desgravamen["TIPDOCUMENTO"].astype(str)
    df_desgravamen['NUMDOCUMENTO'] = df_desgravamen['NUMDOCUMENTO'].astype(str)
    df_desgravamen['NOMBRE_DE_ARCHIVO'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].astype(str)
    df_desgravamen['LINEA_TRAMA'] = df_desgravamen['LINEA_TRAMA'].astype(str)
    df_desgravamen['ORIGEN_ERROR'] = df_desgravamen['ORIGEN_ERROR'].astype(str)
    df_desgravamen['CODIGO_ERROR'] = df_desgravamen['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")

    # Crear nuevas columnas cextrayendo la fecha del nombre de archivo
    df_desgravamen= df_desgravamen[df_desgravamen['PRIMABRUTACAN']<30000]   #Eliminar las filas con valores atipicos muy altos
    df_desgravamen['FECHA_TRAMA'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].apply(extraer_fecha)
    df_desgravamen['FECHA_TRAMA']= pd.to_datetime(df_desgravamen['FECHA_TRAMA'], errors='coerce')
    # Columna para diferenciar el periodo de reporte de errores
    df_desgravamen['FECHA_PERIODO']= pd.to_datetime(FECHA_PERIODO)
    # Crear columna evaluando condiciones en el contenido de otras columnas
    df_desgravamen['TIPO_ERROR']= df_desgravamen.apply(clasificar, axis=1)

    #Extraer datos de la la Trama Original
    df_desgravamen["TIPO_SEGURO"] = df_desgravamen["LINEA_TRAMA"].str[:3]
    df_desgravamen["TIPO_MOVIMIENTO_NUM"] = df_desgravamen["LINEA_TRAMA"].str[48]
    df_desgravamen["TIPO_REGISTRO"] = df_desgravamen["LINEA_TRAMA"].str[43:45]

    # Cruzar con la Tabla de errores
    df_desgravamen= df_desgravamen.merge(df_errores, on='CODIGO_ERROR', how='left')

    # Guardar tabla en BigQuery
    Guardar_en_BigQuery(df_desgravamen, DATASET_ID, TABLE_ID, schema_desgramen)
    print(f"##### EL ARCHIVO: {blob.name} SE HA GUARDADO CORRECTAMENTE EN BIGQUERY #####")



..... CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P1.xlsx ....
##### EL ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P1.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY #####
..... CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P10.xlsx ....
##### EL ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P10.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY #####
..... CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P11.xlsx ....
##### EL ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P11.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY #####
..... CARGAN

--------------

In [4]:
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

In [5]:
bucket = storage_client.bucket(BUCKET_NAME)
blob_errores = bucket.blob('data_entries/REPORTES DE ERRORES/tabla_errores2.xlsx')

In [28]:
df_errores= pd.read_excel(BytesIO(blob_errores.download_as_string()), sheet_name='Sheet1', dtype={'CODIGO_ERROR': str, 'IDEERROR': str})
df_errores.loc[df_errores['CODIGO_ERROR'].isna(), 'CODIGO_ERROR'] = ""
df_errores.loc[df_errores['IDEERROR'].isna(), 'IDEERROR'] = ""
#df_errores.drop(['IDEERROR'], axis=1, inplace=True)
df_errores.head(3)

,IDEERROR,CODIGO_ERROR,DESCRIPCION_ERROR,DETALLE,SOLUCION,SOLUCION_2
0,0974,1068,YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO...,YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO...,CONFIGURACION SAS,ANULAR MASIVAS
1,0005,0003,ERROR EN CARGA DE TRAMA,ERROR EN CARGA DE TRAMA,ANALISIS EMISOR,NaN
2,1158,1155,TRAMA REPETIDA O DUPLICADA,TRAMA REPETIDA O DUPLICADA,CONFIGURACION SAS,ANULAR MASIVAS


In [29]:
bucket = storage_client.bucket(BUCKET_NAME)
blobs_excels = list(bucket.list_blobs(prefix=FOLDER_PATH))

In [30]:
for blob in blobs_excels:
  if blob.name.endswith(".xlsx"):
    #file = blob.download_as_string()
    #excel_file = pd.ExcelFile(file)
    print(blob.name)
    #print(excel_file.sheet_names)

data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P1.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P10.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P11.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P12.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P13.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P14.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P15.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P16.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos

In [ ]:
#blob_consuer = bucket.blob('desgravamen_prestamos/desgravamen - consuer.csv')
#file = blob_consuer.download_as_string()
#df_desgravamen= pd.read_csv(BytesIO(file), dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str, 'IDELOTE': str, 'CODIGO ERROR': str})

In [31]:
print(blobs_excels[3].name)
file = blobs_excels[3].download_as_string()
df_desgravamen= pd.read_excel(BytesIO(file), sheet_name='Hoja1', dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str,
                                                                                           'IDELOTE': str, 'CODIGO ERROR': str,
                                                                                           'SUMA ASEGURADA':str, 'TASA': str,
                                                                                           'TASA RECARGO': str, 'PRIMABRUTACAN':str, 'PRIMANETACAN': str})


data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/16-junio-2025/446 - Desgravamen Desempleo - BBVA - P11.xlsx


In [32]:
df_desgravamen.head(3)

,NRO LOTE,LOTES ANTERIORES,FECHA CARGA,CODIGO PRODUCTO,PRODUCTO,CODIGO PLAN,NOMBRE DE PLAN,COD DE CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO MOVIMIENTO,FEC. INICIO,FEC. FIN,MONEDA,SUMA ASEGURADA,TASA,TASA RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE DE ARCHIVO,IDELOTE,IDEDET.1,ORIGEN ERROR,CODIGO ERROR,DESCRIPCION ERROR,Unnamed: 32
0,1072346,NaN,2023-11-06 15:19:56.000,3827.0,Desgravamen + Desempleo BBVA,380668.0,Plan Premier PEN,00110246654000530585,2023-07-17 00:00:00.000,2027-07-30 00:00:00.000,641167626,Renovacion,2023-07-17,2023-10-30,SOL,22248.4,0.5511,0.5511,11.44,11.11,NaN,NaN,NaN,NaN,NaN,NaN,20100130204_0157001_20231101_003.TXT,951001102466540005305850011024660020054636701P...,1072346,641167626,Error canal,7578,LA FECHA DE INICIO DE VIGENCIA DEL PAGO ES DI...
1,1072346,NaN,2023-11-06 15:19:56.000,3827.0,Desgravamen + Desempleo BBVA,380668.0,Plan Premier PEN,00110246654000530585,2023-07-17 00:00:00.000,2027-07-30 00:00:00.000,641167626,Renovacion,2023-07-17,2023-10-30,SOL,22248.4,0.5511,0.5511,11.44,11.11,NaN,NaN,NaN,NaN,NaN,NaN,20100130204_0157001_20231101_003.TXT,951001102466540005305850011024660020054636701P...,1072346,641167626,Error Rimac,1157,EL ALTA DEL NRO. CERTIFICADO EXTERNO: 00110246...
2,1072346,NaN,2023-11-06 15:19:56.000,3827.0,Desgravamen + Desempleo BBVA,380668.0,Plan Premier PEN,00110785324002691714,2023-04-03 00:00:00.000,2026-04-06 00:00:00.000,641168260,Renovacion,2023-09-05,2023-10-05,SOL,2939.88,0.5511,0.5511,1.62,1.57,NaN,NaN,NaN,NaN,NaN,NaN,20100130204_0157001_20231101_003.TXT,951001107853240026917140011081415022345778401P...,1072346,641168260,Error canal,11,EL CAMPO INDPERIODOPAGO DEL(A) CERTIFICADO NO ...


In [33]:
df_desgravamen.columns = (
    df_desgravamen.columns
    .str.strip()  # quitar espacios al inicio/fin
    .str.upper()  # opcional: todo en mayúsculas
    .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)
#df_desgravamen.drop(['IDELOTE','IDEDET_1'], axis=1, inplace=True)
#df_desgravamen= df_desgravamen.rename(columns={'DESCRIPCION_ERROR':'DESCRIPCION_ERROR_SAS'})
df_desgravamen.drop(['IDEDET_1','ORIGEN_ERROR','UNNAMED__32'], axis=1, inplace=True)
df_desgravamen= df_desgravamen.rename(columns={'IDELOTE': 'LINEA_TRAMA','CODIGO_ERROR':'ORIGEN_ERROR', 'DESCRIPCION_ERROR': 'CODIGO_ERROR'})

In [34]:
df_desgravamen["MONEDA"] = df_desgravamen["MONEDA"].replace(r"^\s*$", None, regex=True)
df_desgravamen["MONEDA"] = df_desgravamen["MONEDA"].where(df_desgravamen["MONEDA"].isin(["USD", "SOL"]), "SIN DATO")

In [35]:
df_desgravamen= df_desgravamen.fillna({'LOTES_ANTERIORES':'', 'ORIGEN_ERROR': 'SIN DATO', 'TIPDOCUMENTO':'SIN DATO'})
df_desgravamen.loc[df_desgravamen['ORIGEN_ERROR'] == 'nan', 'ORIGEN_ERROR'] = 'SIN DATO'

In [36]:
df_desgravamen['SUMA_ASEGURADA'] = df_desgravamen['SUMA_ASEGURADA'].str.replace(',', '.', regex=False)
df_desgravamen['TASA'] = df_desgravamen['TASA'].str.replace(',', '.', regex=False)
df_desgravamen['TASA_RECARGO'] = df_desgravamen['TASA_RECARGO'].str.replace(',', '.', regex=False)
df_desgravamen['PRIMABRUTACAN'] = df_desgravamen['PRIMABRUTACAN'].str.replace(',', '.', regex=False)
df_desgravamen['PRIMANETACAN'] = df_desgravamen['PRIMANETACAN'].str.replace(',', '.', regex=False)

In [37]:
df_desgravamen.head()

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,CODIGO_ERROR
0,1072346,,2023-11-06 15:19:56.000,3827.0,Desgravamen + Desempleo BBVA,380668.0,Plan Premier PEN,00110246654000530585,2023-07-17 00:00:00.000,2027-07-30 00:00:00.000,641167626,Renovacion,2023-07-17,2023-10-30,SOL,22248.4,0.5511,0.5511,11.44,11.11,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0157001_20231101_003.TXT,951001102466540005305850011024660020054636701P...,Error canal,7578
1,1072346,,2023-11-06 15:19:56.000,3827.0,Desgravamen + Desempleo BBVA,380668.0,Plan Premier PEN,00110246654000530585,2023-07-17 00:00:00.000,2027-07-30 00:00:00.000,641167626,Renovacion,2023-07-17,2023-10-30,SOL,22248.4,0.5511,0.5511,11.44,11.11,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0157001_20231101_003.TXT,951001102466540005305850011024660020054636701P...,Error Rimac,1157
2,1072346,,2023-11-06 15:19:56.000,3827.0,Desgravamen + Desempleo BBVA,380668.0,Plan Premier PEN,00110785324002691714,2023-04-03 00:00:00.000,2026-04-06 00:00:00.000,641168260,Renovacion,2023-09-05,2023-10-05,SOL,2939.88,0.5511,0.5511,1.62,1.57,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0157001_20231101_003.TXT,951001107853240026917140011081415022345778401P...,Error canal,11
3,1072346,,2023-11-06 15:19:56.000,3827.0,Desgravamen + Desempleo BBVA,380668.0,Plan Premier PEN,00110785324002691714,2023-04-03 00:00:00.000,2026-04-06 00:00:00.000,641168260,Renovacion,2023-09-05,2023-10-05,SOL,2939.88,0.5511,0.5511,1.62,1.57,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0157001_20231101_003.TXT,951001107853240026917140011081415022345778401P...,Error canal,2395
4,1072346,,2023-11-06 15:19:56.000,3827.0,Desgravamen + Desempleo BBVA,380668.0,Plan Premier PEN,00110785324002691714,2023-04-03 00:00:00.000,2026-04-06 00:00:00.000,641168260,Renovacion,2023-09-05,2023-10-05,SOL,2939.88,0.5511,0.5511,1.62,1.57,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0157001_20231101_003.TXT,951001107853240026917140011081415022345778401P...,Error canal,7578


In [38]:
df_desgravamen[df_desgravamen['MONEDA'].isna()].head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,CODIGO_ERROR


In [39]:
def limpiar_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

In [40]:
df_desgravamen["LOTES_ANTERIORES"] = df_desgravamen["LOTES_ANTERIORES"].astype(str)
df_desgravamen['FECHA_CARGA'] = pd.to_datetime(df_desgravamen['FECHA_CARGA'], format="%d/%m/%Y %I:%M:%S %p", errors='coerce')
df_desgravamen['CODIGO_PRODUCTO'] = df_desgravamen['CODIGO_PRODUCTO'].fillna(0).astype('int')
df_desgravamen["PRODUCTO"] = df_desgravamen["PRODUCTO"].astype(str)
df_desgravamen['CODIGO_PLAN'] = df_desgravamen['CODIGO_PLAN'].fillna(0).astype('int')
df_desgravamen["NOMBRE_DE_PLAN"] = df_desgravamen["NOMBRE_DE_PLAN"].astype(str)
df_desgravamen["COD_DE_CERTIFICADO"] = df_desgravamen["COD_DE_CERTIFICADO"].astype(str)
df_desgravamen['FECINI_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECINI_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['FECFIN_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECFIN_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen["TIPO_MOVIMIENTO"] = df_desgravamen["TIPO_MOVIMIENTO"].astype(str)
df_desgravamen['FEC__INICIO'] = pd.to_datetime(df_desgravamen['FEC__INICIO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['FEC__FIN'] = pd.to_datetime(df_desgravamen['FEC__FIN'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['MONEDA'] = df_desgravamen['MONEDA'].astype(str)
df_desgravamen['SUMA_ASEGURADA'] = pd.to_numeric(df_desgravamen['SUMA_ASEGURADA'], errors="coerce").astype('float64')
df_desgravamen['TASA'] = pd.to_numeric(df_desgravamen['TASA'], errors="coerce").astype('float64')
df_desgravamen['TASA_RECARGO'] = pd.to_numeric(df_desgravamen['TASA_RECARGO'], errors="coerce").astype('float64')
df_desgravamen['PRIMABRUTACAN'] = pd.to_numeric(df_desgravamen['PRIMABRUTACAN'], errors="coerce").astype('float64')
df_desgravamen['PRIMANETACAN'] = pd.to_numeric(df_desgravamen['PRIMANETACAN'], errors="coerce").astype('float64')
df_desgravamen['NOMCOMPLETO'] = df_desgravamen['NOMCOMPLETO'].astype(str)
df_desgravamen['APEPATERNO'] = df_desgravamen['APEPATERNO'].astype(str)
df_desgravamen['APEMATERNO'] = df_desgravamen['APEMATERNO'].astype(str)
df_desgravamen["FECNACIMIENTO"] = limpiar_fecha(df_desgravamen["FECNACIMIENTO"])
df_desgravamen["TIPDOCUMENTO"] = df_desgravamen["TIPDOCUMENTO"].astype(str)
df_desgravamen['NUMDOCUMENTO'] = df_desgravamen['NUMDOCUMENTO'].astype(str)
df_desgravamen['NOMBRE_DE_ARCHIVO'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].astype(str)
df_desgravamen['LINEA_TRAMA'] = df_desgravamen['LINEA_TRAMA'].astype(str)
df_desgravamen['ORIGEN_ERROR'] = df_desgravamen['ORIGEN_ERROR'].astype(str)
#df_desgravamen['IDEERROR'] = df_desgravamen['IDEERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")
#df_desgravamen['CODIGO_ERROR'] = df_desgravamen['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else np.nan)
df_desgravamen['CODIGO_ERROR'] = df_desgravamen['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")

In [20]:
df_desgravamen['CODIGO_PRODUCTO'].value_counts()

,count
CODIGO_PRODUCTO,
3827,57987
0,1027


In [41]:
#### FUNCION PARA EXTRAER LA FECHA DEL NOMBRE DE ARCHIVO
def extraer_fecha(fecha):
    try:
        fecha_str = fecha.split("_")[2]
        return datetime.strptime(fecha_str, "%Y%m%d").date()
    except Exception:
        return None  # En caso de error

#### FUNCION PARA CLASIFICAR EL REGISTRO
def clasificar(row):
    # Si lote no es nulo
    if str(row['LOTES_ANTERIORES']).strip() != "":
        return 'Regularizacion no Exitosa'

    # Si último número del nombre de archivo > 100
    try:
        ultimo_numero = int(row['NOMBRE_DE_ARCHIVO'].split("_")[-1][:-4])
        if ultimo_numero > 100:
            return 'Regularizacion no Exitosa'
    except:
        pass

    # Si ninguna condición se cumple
    return 'Mes corriente'

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f'----- Se ha creado la tabla {table_id} en el dataset {dataset_id} -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return

In [42]:
df_desgravamen= df_desgravamen[df_desgravamen['PRIMABRUTACAN']<30000]   #Eliminar las filas con valores atipicos muy altos
df_desgravamen['FECHA_TRAMA'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].apply(extraer_fecha)    # Crear nuevas columnas cextrayendo la fecha del nombre de archivo
df_desgravamen['FECHA_TRAMA']= pd.to_datetime(df_desgravamen['FECHA_TRAMA'], errors='coerce')
df_desgravamen['FECHA_PERIODO']= pd.to_datetime(FECHA_PERIODO)
df_desgravamen['TIPO_ERROR']= df_desgravamen.apply(clasificar, axis=1)    # Crear colunma evaluando condiciones en el contenido de otras columnas

In [43]:
df_desgravamen["TIPO_SEGURO"] = df_desgravamen["LINEA_TRAMA"].str[:3]
df_desgravamen["TIPO_MOVIMIENTO_NUM"] = df_desgravamen["LINEA_TRAMA"].str[48]
df_desgravamen["TIPO_REGISTRO"] = df_desgravamen["LINEA_TRAMA"].str[43:45]

In [24]:
df_desgravamen[df_desgravamen['CODIGO_ERROR']== ''].head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,CODIGO_ERROR,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR,TIPO_SEGURO,TIPO_MOVIMIENTO_NUM,TIPO_REGISTRO


In [45]:
#df_desgravamen= df_desgravamen.merge(df_errores, on='IDEERROR', how='left')
#df_desgravamen.drop(['IDEERROR'], axis=1, inplace=True)
df_desgravamen= df_desgravamen.merge(df_errores, on='CODIGO_ERROR', how='left')
df_desgravamen.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,CODIGO_ERROR,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR,TIPO_SEGURO,TIPO_MOVIMIENTO_NUM,TIPO_REGISTRO,IDEERROR,DESCRIPCION_ERROR,DETALLE,SOLUCION,SOLUCION_2
0,1072346,,NaT,3827,Desgravamen + Desempleo BBVA,380668,Plan Premier PEN,00110246654000530585,NaT,NaT,641167626,Renovacion,2023-07-17,2023-10-30,SOL,22248.40,0.5511,0.5511,11.44,11.11,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0157001_20231101_003.TXT,951001102466540005305850011024660020054636701P...,Error canal,7578,2023-11-01,2025-06-16,Mes corriente,951,4,01,2046,HUECOS DE VIGENCIA,HUECOS DE VIGENCIA,CONFIGURACION SAS,SOLUCION COMPLEJA
1,1072346,,NaT,3827,Desgravamen + Desempleo BBVA,380668,Plan Premier PEN,00110246654000530585,NaT,NaT,641167626,Renovacion,2023-07-17,2023-10-30,SOL,22248.40,0.5511,0.5511,11.44,11.11,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0157001_20231101_003.TXT,951001102466540005305850011024660020054636701P...,Error Rimac,1157,2023-11-01,2025-06-16,Mes corriente,951,4,01,1156,ERROR EN ALTA,EL ALTA DEL NRO. CERTIFICADO EXTERNO SE ENCUEN...,ANALISIS EMISOR,NaN
2,1072346,,NaT,3827,Desgravamen + Desempleo BBVA,380668,Plan Premier PEN,00110785324002691714,NaT,NaT,641168260,Renovacion,2023-09-05,2023-10-05,SOL,2939.88,0.5511,0.5511,1.62,1.57,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0157001_20231101_003.TXT,951001107853240026917140011081415022345778401P...,Error canal,0011,2023-11-01,2025-06-16,Mes corriente,951,4,01,0014,CAMPOS EN BLANCO,CAMPOS EN BLANCO,ANALISIS EMISOR,NaN


In [46]:
df_desgravamen['DESCRIPCION_ERROR'].value_counts()

,count
DESCRIPCION_ERROR,
HUECOS DE VIGENCIA,33780
VALIDACIONES CONSECUENCIA ACSEL E,19122
ERROR DE TASA,4013
ERROR EN CARGA DE TRAMA,1018
ERROR EN ALTA,974
CAMPOS EN BLANCO,83
VALIDACIONES CONSECUENCIAS,20
TRAMA REPETIDA O DUPLICADA,9
YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO DE VIGENCIA,3


In [ ]:
df_desgravamen["DESCRIPCION_ERROR"] = df_desgravamen["DESCRIPCION_ERROR"].astype(str)
df_desgravamen["DETALLE"] = df_desgravamen["DETALLE"].astype(str)
df_desgravamen["SOLUCION"] = df_desgravamen["SOLUCION"].astype(str)
df_desgravamen["SOLUCION_2"] = df_desgravamen["SOLUCION_2"].astype(str)

In [47]:
df_desgravamen.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59024 entries, 0 to 59023
Data columns (total 41 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   NRO_LOTE                 59024 non-null  int64         
 1   LOTES_ANTERIORES         59024 non-null  object        
 2   FECHA_CARGA              0 non-null      datetime64[ns]
 3   CODIGO_PRODUCTO          59024 non-null  int64         
 4   PRODUCTO                 59024 non-null  object        
 5   CODIGO_PLAN              59024 non-null  int64         
 6   NOMBRE_DE_PLAN           59024 non-null  object        
 7   COD_DE_CERTIFICADO       59024 non-null  object        
 8   FECINI_ALTA_CERTIFICADO  0 non-null      datetime64[ns]
 9   FECFIN_ALTA_CERTIFICADO  0 non-null      datetime64[ns]
 10  IDEDET                   59024 non-null  int64         
 11  TIPO_MOVIMIENTO          59024 non-null  object        
 12  FEC__INICIO              59024 n

In [49]:
TABLE_ID= "ERRORES_desgravamen_prestamos_temp"
schema_desgramen = [
        bigquery.SchemaField("NRO_LOTE", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("LOTES_ANTERIORES", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_PLAN", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("NOMBRE_DE_PLAN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("COD_DE_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECINI_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECFIN_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("IDEDET", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("TIPO_MOVIMIENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_FIN ", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SUMA_ASEGURADA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA_RECARGO", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMABRUTACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMANETACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("NOMCOMPLETO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEPATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEMATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECNACIMIENTO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_DE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("LINEA_TRAMA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("ORIGEN_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR_SAS", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_TRAMA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_PERIODO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_SEGURO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_MOVIMIENTO_NUM", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_REGISTRO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("IDEERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DETALLE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION_2", bigquery.enums.SqlTypeNames.STRING)
    ]
Guardar_en_BigQuery(df_desgravamen, DATASET_ID, TABLE_ID, schema_desgramen)

----- REGISTROS AGREGADOS CORRECTAMENTE EN: ERRORES_desgravamen_prestamos_temp -------
